In [1]:
import pandas as pd
import numpy as np

from mlxtend.frequent_patterns import apriori, association_rules

c:\Users\15195\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
# Read the transaction item table and cluster results.
transaction_product = pd.read_csv(
    r"C:\Users\15195\Desktop\coding part\dataset merged\transaction_product_merged.csv"
)

clustered_households = pd.read_csv(
    r"C:\Users\15195\Desktop\coding part\k-means-修改版\clustered_household_features (k=3).csv"
)

print("Transaction-product shape:", transaction_product.shape)
print("Clustered households shape:", clustered_households.shape)

print(transaction_product.columns.tolist())
print(clustered_households.columns.tolist())

Transaction-product shape: (2595732, 18)
Clustered households shape: (2500, 19)
['household_key', 'BASKET_ID', 'DAY', 'PRODUCT_ID', 'QUANTITY', 'SALES_VALUE', 'STORE_ID', 'RETAIL_DISC', 'TRANS_TIME', 'WEEK_NO', 'COUPON_DISC', 'COUPON_MATCH_DISC', 'MANUFACTURER', 'DEPARTMENT', 'BRAND', 'COMMODITY_DESC', 'SUB_COMMODITY_DESC', 'CURR_SIZE_OF_PRODUCT']
['household_key', 'recency', 'frequency', 'monetary', 'total_quantity', 'unique_products', 'category_diversity', 'first_purchase_day', 'last_purchase_day', 'average_basket_value', 'average_basket_size', 'customer_lifetime_days', 'total_retail_discount', 'total_coupon_discount', 'total_coupon_match_discount', 'total_discount', 'discount_ratio', 'coupon_discount_ratio', 'cluster']


In [3]:
# Extract the cluster labels for households
cluster_labels = (
    clustered_households[
        ["household_key", "cluster"]
    ]
    .drop_duplicates(subset=["household_key"])
)

print(cluster_labels.head())

print(
    cluster_labels["cluster"]
    .value_counts()
    .sort_index()
)

   household_key  cluster
0              1        1
1              2        1
2              3        1
3              4        2
4              5        2
cluster
0     286
1    1660
2     554
Name: count, dtype: int64


In [4]:
# Merge the clustering labels back into the transaction product table
transaction_product_with_segment = transaction_product.merge(
    cluster_labels,
    on="household_key",
    how="inner",
    validate="many_to_one"
)

print("Merged shape:", transaction_product_with_segment.shape)

print(
    transaction_product_with_segment[
        [
            "household_key",
            "BASKET_ID",
            "PRODUCT_ID",
            "COMMODITY_DESC",
            "cluster"
        ]
    ].head()
)

print(
    transaction_product_with_segment["cluster"]
    .value_counts()
    .sort_index()
)

Merged shape: (2595732, 19)
   household_key    BASKET_ID  PRODUCT_ID               COMMODITY_DESC  \
0           2375  26984851472     1004906                     POTATOES   
1           2375  26984851472     1033142                       ONIONS   
2           2375  26984851472     1036325      VEGETABLES - ALL OTHERS   
3           2375  26984851472     1082185               TROPICAL FRUIT   
4           2375  26984851472     8160430  ORGANICS FRUIT & VEGETABLES   

   cluster  
0        1  
1        1  
2        1  
3        1  
4        1  
cluster
0     723767
1    1787219
2      84746
Name: count, dtype: int64


In [5]:
# Clean up the shopping cart data
# The purpose of the cleanup is to aim to retain: valid BASKET_ID, valid COMMODITY_DESC, valid cluster, positive sales amount, positive purchase quantity
basket_source = transaction_product_with_segment.copy()

basket_source = basket_source.dropna(
    subset=[
        "household_key",
        "BASKET_ID",
        "COMMODITY_DESC",
        "cluster"
    ]
)

basket_source = basket_source[
    (basket_source["SALES_VALUE"] > 0) &
    (basket_source["QUANTITY"] > 0)
].copy()

basket_source["COMMODITY_DESC"] = (
    basket_source["COMMODITY_DESC"]
    .astype(str)
    .str.strip()
    .str.upper()
)

basket_source["cluster"] = (
    basket_source["cluster"]
    .astype(int)
)

print("Valid baskets:", basket_source["BASKET_ID"].nunique())
print("Commodity categories:", basket_source["COMMODITY_DESC"].nunique())
print("Clusters:", sorted(basket_source["cluster"].unique()))

Valid baskets: 275539
Commodity categories: 307
Clusters: [np.int64(0), np.int64(1), np.int64(2)]


In [6]:
# Examine sparsity at the individual product level

total_baskets = basket_source["BASKET_ID"].nunique()

product_frequency = (
    basket_source
    .groupby("PRODUCT_ID")["BASKET_ID"]
    .nunique()
)

total_products = product_frequency.size

one_percent_threshold = total_baskets * 0.01

products_below_1pct = (
    product_frequency < one_percent_threshold
).sum()

products_below_1pct_share = (
    products_below_1pct / total_products * 100
)

print("Total valid baskets:", total_baskets)
print("Total unique products:", total_products)
print("1% basket threshold:", one_percent_threshold)
print(
    "Products appearing in fewer than 1% of baskets:",
    products_below_1pct
)
print(
    "Percentage of products appearing in fewer than 1% of baskets:",
    round(products_below_1pct_share, 2),
    "%"
)

Total valid baskets: 275539
Total unique products: 91905
1% basket threshold: 2755.39
Products appearing in fewer than 1% of baskets: 91857
Percentage of products appearing in fewer than 1% of baskets: 99.95 %
